# Training

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Load your data
df = pd.read_pickle("../data/processed/preprocessed_weather.pkl")

# --- Preprocessing ---

# Convert time to useful numeric features
df["time"] = pd.to_datetime(df["time"])
df["year"] = df["time"].dt.year
df["month"] = df["time"].dt.month
df["day"] = df["time"].dt.day

# Drop original time column
df = df.drop(columns=["time"])

# Convert categorical to numeric (one-hot encoding)
df = pd.get_dummies(df, columns=["weather_code"], drop_first=True)

df.head()

,temperature_2m_max,temperature_2m_min,temperature_2m_mean,apparent_temperature_max,apparent_temperature_min,relative_humidity_2m_max,relative_humidity_2m_min,relative_humidity_2m_mean,wind_speed_10m_max,wind_gusts_10m_max,...,day,weather_code_1,weather_code_2,weather_code_3,weather_code_51,weather_code_53,weather_code_55,weather_code_61,weather_code_63,weather_code_65
0,21.8,6.5,14.3,20.9,4.3,94,39,68,4.9,11.4,...,1,False,False,False,False,False,False,False,False,False
1,22.6,8.0,14.6,21.7,6.0,95,41,71,6.4,9.8,...,2,False,False,False,False,False,False,False,False,False
2,22.6,7.8,14.6,21.2,6.2,97,41,73,6.5,13.4,...,3,False,False,False,False,False,False,False,False,False
3,22.2,8.0,14.7,21.6,6.1,96,35,69,5.8,10.3,...,4,False,False,True,False,False,False,False,False,False
4,23.4,9.9,16.0,21.5,8.2,86,42,66,7.9,16.6,...,5,False,False,True,False,False,False,False,False,False


## Linear Regression

In [2]:
# Set up training loop with Weights and Biases

from sklearn.linear_model import LinearRegression, Ridge, Lasso
import joblib
import os
import tempfile
import wandb

training_params = [
    # --- LinearRegression: only knob worth turning is fit_intercept ---
    {
        "run_name": "linear_reg_no_intercept",
        "model_class": LinearRegression,
        "params": {"fit_intercept": False}
    },
    {
        "run_name": "linear_reg_with_intercept",
        "model_class": LinearRegression,
        "params": {"fit_intercept": True}
    },

    # --- Ridge: L2 regularization, sweep alpha (regularization strength) ---
    {
        "run_name": "ridge_alpha_0.1",
        "model_class": Ridge,
        "params": {"alpha": 0.1, "fit_intercept": True}
    },
    {
        "run_name": "ridge_alpha_1.0",
        "model_class": Ridge,
        "params": {"alpha": 1.0, "fit_intercept": True}
    },
    {
        "run_name": "ridge_alpha_10.0",
        "model_class": Ridge,
        "params": {"alpha": 10.0, "fit_intercept": True}
    },

    # --- Lasso: L1 regularization, also sweeps alpha ---
    {
        "run_name": "lasso_alpha_0.1",
        "model_class": Lasso,
        "params": {"alpha": 0.1, "fit_intercept": True, "max_iter": 5000}
    },
    {
        "run_name": "lasso_alpha_1.0",
        "model_class": Lasso,
        "params": {"alpha": 1.0, "fit_intercept": True, "max_iter": 5000}
    },
]

# Prepare data
X = df.drop(columns=["apparent_temperature_max"], axis=1)
y = df["apparent_temperature_max"]

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Run each of the params with Weights and Biases
for config in training_params:
    run_name = config["run_name"]
    params = config["params"]
    model_class = config["model_class"]

    with wandb.init(project="weather-prediction", name=run_name, config=params) as run:
        model = model_class(**params)   # <-- use model_class here
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        mse = mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        run.log({"MSE": mse, "R2": r2})

        with tempfile.TemporaryDirectory() as tmpdir:
            model_path = os.path.join(tmpdir, "model.pkl")
            joblib.dump(model, model_path)
            artifact = wandb.Artifact(
                name=f"{run_name.lower()}_estimator",
                type="model",
                metadata={"MSE": float(mse), "R2": float(r2), "run_name": run_name},
            )
            artifact.add_file(model_path, name="model.pkl")
            run.log_artifact(artifact)

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /Users/srihariraman/.netrc
wandb: Currently logged in as: srihariraman9 (weatherml) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


MSE,▁
R2,▁
MSE,0.45607
R2,0.99224


MSE,▁
R2,▁
MSE,0.45953
R2,0.99219


MSE,▁
R2,▁
MSE,0.45962
R2,0.99218


MSE,▁
R2,▁
MSE,0.4607
R2,0.99217


MSE,▁
R2,▁
MSE,0.47092
R2,0.99199


MSE,▁
R2,▁
MSE,0.49508
R2,0.99158


MSE,▁
R2,▁
MSE,0.94337
R2,0.98396


## KNN Regressor

In [3]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import os
import tempfile
import wandb

knn_training_params = [
    # --- Sweep n_neighbors with uniform weighting ---
    {"run_name": "knn_k3_uniform", "params": {"n_neighbors": 3, "weights": "uniform", "metric": "euclidean", "algorithm": "auto"}},
    {"run_name": "knn_k5_uniform", "params": {"n_neighbors": 5, "weights": "uniform", "metric": "euclidean", "algorithm": "auto"}},
    {"run_name": "knn_k10_uniform", "params": {"n_neighbors": 10, "weights": "uniform", "metric": "euclidean", "algorithm": "auto"}},
    {"run_name": "knn_k20_uniform", "params": {"n_neighbors": 20, "weights": "uniform", "metric": "euclidean", "algorithm": "auto"}},
    {"run_name": "knn_k50_uniform", "params": {"n_neighbors": 50, "weights": "uniform", "metric": "euclidean", "algorithm": "auto"}},

    # --- Sweep n_neighbors with distance weighting ---
    {"run_name": "knn_k3_distance", "params": {"n_neighbors": 3, "weights": "distance", "metric": "euclidean", "algorithm": "auto"}},
    {"run_name": "knn_k5_distance", "params": {"n_neighbors": 5, "weights": "distance", "metric": "euclidean", "algorithm": "auto"}},
    {"run_name": "knn_k10_distance", "params": {"n_neighbors": 10, "weights": "distance", "metric": "euclidean", "algorithm": "auto"}},
    {"run_name": "knn_k20_distance", "params": {"n_neighbors": 20, "weights": "distance", "metric": "euclidean", "algorithm": "auto"}},
    {"run_name": "knn_k50_distance", "params": {"n_neighbors": 50, "weights": "distance", "metric": "euclidean", "algorithm": "auto"}},

    # --- Sweep distance metrics ---
    {"run_name": "knn_k10_manhattan", "params": {"n_neighbors": 10, "weights": "distance", "metric": "manhattan", "algorithm": "auto"}},
    {"run_name": "knn_k10_chebyshev", "params": {"n_neighbors": 10, "weights": "distance", "metric": "chebyshev", "algorithm": "auto"}},
    {"run_name": "knn_k10_minkowski_p3", "params": {"n_neighbors": 10, "weights": "distance", "metric": "minkowski", "metric_params": {"p": 3}, "algorithm": "auto"}},

    # --- Sweep algorithms ---
    {"run_name": "knn_k10_balltree", "params": {"n_neighbors": 10, "weights": "distance", "metric": "euclidean", "algorithm": "ball_tree"}},
    {"run_name": "knn_k10_kdtree", "params": {"n_neighbors": 10, "weights": "distance", "metric": "euclidean", "algorithm": "kd_tree"}},
]

# KNN-specific training loop (with scaling pipeline)
for config in knn_training_params:
    run_name = config["run_name"]
    params = config["params"]

    with wandb.init(project="weather-prediction", name=run_name, config=params) as run:

        # Build pipeline — scaler is essential for KNN
        knn_model = KNeighborsRegressor(**params)
        
        pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("model", knn_model)
        ])

        pipeline.fit(X_train, y_train)

        y_pred = pipeline.predict(X_test)
        mse = mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        run.log({"MSE": mse, "R2": r2})

        with tempfile.TemporaryDirectory() as tmpdir:
            model_path = os.path.join(tmpdir, "model.pkl")
            joblib.dump(pipeline, model_path)
            artifact = wandb.Artifact(
                name=f"{run_name.lower()}_estimator",
                type="model",
                metadata={"MSE": float(mse), "R2": float(r2), "run_name": run_name},
            )
            artifact.add_file(model_path, name="model.pkl")
            run.log_artifact(artifact)

MSE,▁
R2,▁
MSE,4.34275
R2,0.92615


MSE,▁
R2,▁
MSE,4.19748
R2,0.92862


MSE,▁
R2,▁
MSE,4.32482
R2,0.92645


MSE,▁
R2,▁
MSE,5.07167
R2,0.91375


MSE,▁
R2,▁
MSE,6.04382
R2,0.89722


...

MSE,▁
R2,▁
MSE,10.82586
R2,0.8159


/Users/srihariraman/Desktop/Coding Projects/Classes/Tutoring/Nathan/WeatherML/.venv/lib/python3.12/site-packages/sklearn/neighbors/_regression.py:227: SyntaxWarning: Parameter p is found in metric_params. The corresponding parameter from __init__ is ignored.
  return self._fit(X, y)


MSE,▁
R2,▁
MSE,5.22582
R2,0.91113


MSE,▁
R2,▁
MSE,3.95874
R2,0.93268


MSE,▁
R2,▁
MSE,3.95874
R2,0.93268


# Polynomial Regression

In [4]:
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import numpy as np
import tempfile
import wandb

import sys
import os

sys.path.append(os.path.abspath(".."))

poly_training_params = [
    # --- Degree sweep with fixed alpha ---
    {"run_name": "poly_deg1_ridge1.0",  "params": {"degree": 1, "alpha": 1.0}},
    {"run_name": "poly_deg2_ridge1.0",  "params": {"degree": 2, "alpha": 1.0}},
    {"run_name": "poly_deg3_ridge1.0",  "params": {"degree": 3, "alpha": 1.0}},

    # --- Alpha sweep at degree 2 ---
    {"run_name": "poly_deg2_ridge0.01",  "params": {"degree": 2, "alpha": 0.01}},
    {"run_name": "poly_deg2_ridge0.1",   "params": {"degree": 2, "alpha": 0.1}},
    {"run_name": "poly_deg2_ridge10.0",  "params": {"degree": 2, "alpha": 10.0}},
    {"run_name": "poly_deg2_ridge100.0", "params": {"degree": 2, "alpha": 100.0}},

    # --- Alpha sweep at degree 3 ---
    {"run_name": "poly_deg3_ridge0.1",   "params": {"degree": 3, "alpha": 0.1}},
    {"run_name": "poly_deg3_ridge10.0",  "params": {"degree": 3, "alpha": 10.0}},
    {"run_name": "poly_deg3_ridge100.0", "params": {"degree": 3, "alpha": 100.0}},

    # --- interaction_only: skip x² terms, keep only cross-features ---
    {"run_name": "poly_deg2_interactions_only", "params": {"degree": 2, "alpha": 1.0, "interaction_only": True}},
    {"run_name": "poly_deg3_interactions_only", "params": {"degree": 3, "alpha": 1.0, "interaction_only": True}},
]

for config in poly_training_params:
    run_name = config["run_name"]
    params   = config["params"]

    # Separate pipeline params from W&B config
    degree           = params["degree"]
    alpha            = params["alpha"]
    interaction_only = params.get("interaction_only", False)

    with wandb.init(project="weather-prediction", name=run_name, config=params) as run:
        pipeline = Pipeline([
            ("poly",   PolynomialFeatures(
                            degree=degree,
                            include_bias=False,
                            interaction_only=interaction_only
                       )),
            ("scaler", StandardScaler()),
            ("model",  Ridge(alpha=alpha, fit_intercept=True)),
        ])
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)

        mse  = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2   = r2_score(y_test, y_pred)

        # Log feature count so W&B lets you see expansion cost
        n_features_out = pipeline.named_steps["poly"].n_output_features_

        run.log({
            "MSE":              mse,
            "R2":               r2,
            "rmse":             rmse,
            "n_features_out":   n_features_out,
        })

        with tempfile.TemporaryDirectory() as tmpdir:
            model_path = os.path.join(tmpdir, "model.pkl")
            joblib.dump(pipeline, model_path)
            artifact = wandb.Artifact(
                name=f"{run_name.lower()}_estimator",
                type="model",
                metadata={"MSE": float(mse), "R2": float(r2), "run_name": run_name},
            )
            artifact.add_file(model_path, name="model.pkl")
            run.log_artifact(artifact)

MSE,▁
R2,▁
n_features_out,▁
rmse,▁
MSE,0.45955
R2,0.99218
n_features_out,29
rmse,0.6779


MSE,▁
R2,▁
n_features_out,▁
rmse,▁
MSE,0.39336
R2,0.99331
n_features_out,464
rmse,0.62719


MSE,▁
R2,▁
n_features_out,▁
rmse,▁
MSE,0.48852
R2,0.99169
n_features_out,4959
rmse,0.69894


MSE,▁
R2,▁
n_features_out,▁
rmse,▁
MSE,0.43039
R2,0.99268
n_features_out,464
rmse,0.65604


MSE,▁
R2,▁
n_features_out,▁
rmse,▁
MSE,0.41454
R2,0.99295
n_features_out,464
rmse,0.64385


...

MSE,▁
R2,▁
n_features_out,▁
rmse,▁
MSE,0.68334
R2,0.98838
n_features_out,4959
rmse,0.82664


MSE,▁
R2,▁
n_features_out,▁
rmse,▁
MSE,0.40902
R2,0.99304
n_features_out,4959
rmse,0.63954


MSE,▁
R2,▁
n_features_out,▁
rmse,▁
MSE,0.40909
R2,0.99304
n_features_out,4959
rmse,0.6396


MSE,▁
R2,▁
n_features_out,▁
rmse,▁
MSE,0.39742
R2,0.99324
n_features_out,435
rmse,0.63041


MSE,▁
R2,▁
n_features_out,▁
rmse,▁
MSE,0.50596
R2,0.9914
n_features_out,4089
rmse,0.71131


# Random Forest Regression

In [5]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import numpy as np
import os
import tempfile
import wandb

# Parameter sweep
rf_training_params = [
    # --- Sweep n_estimators ---
    {"run_name": "rf_n100_depth5",  "params": {"n_estimators": 100, "max_depth": 5,    "min_samples_split": 2,  "max_features": "sqrt"}},
    {"run_name": "rf_n200_depth5",  "params": {"n_estimators": 200, "max_depth": 5,    "min_samples_split": 2,  "max_features": "sqrt"}},
    {"run_name": "rf_n500_depth5",  "params": {"n_estimators": 500, "max_depth": 5,    "min_samples_split": 2,  "max_features": "sqrt"}},

    # --- Sweep max_depth ---
    {"run_name": "rf_n200_depth3",  "params": {"n_estimators": 200, "max_depth": 3,    "min_samples_split": 2,  "max_features": "sqrt"}},
    {"run_name": "rf_n200_depthNone", "params": {"n_estimators": 200, "max_depth": None, "min_samples_split": 2, "max_features": "sqrt"}},
    {"run_name": "rf_n200_depth10", "params": {"n_estimators": 200, "max_depth": 10,   "min_samples_split": 2,  "max_features": "sqrt"}},

    # --- Sweep min_samples_split ---
    {"run_name": "rf_n200_split5",  "params": {"n_estimators": 200, "max_depth": 5,    "min_samples_split": 5,  "max_features": "sqrt"}},
    {"run_name": "rf_n200_split10", "params": {"n_estimators": 200, "max_depth": 5,    "min_samples_split": 10, "max_features": "sqrt"}},

    # --- Sweep max_features ---
    {"run_name": "rf_n200_feat_log2",  "params": {"n_estimators": 200, "max_depth": 5, "min_samples_split": 2, "max_features": "log2"}},
    {"run_name": "rf_n200_feat_none",  "params": {"n_estimators": 200, "max_depth": 5, "min_samples_split": 2, "max_features": 1.0}},
]

for config in rf_training_params:
    run_name = config["run_name"]
    params   = config["params"]

    with wandb.init(project="weather-prediction", name=run_name, config=params) as run:
        model = RandomForestRegressor(
            **params,
            random_state=42,
            n_jobs=-1
        )

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        mse  = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2   = r2_score(y_test, y_pred)

        # Feature importances — useful to inspect in W&B
        importances = dict(zip(X.columns, model.feature_importances_.round(4)))

        run.log({
            "MSE":                 mse,
            "R2":                  r2,
            "rmse":                rmse,
            "feature_importances": importances,
        })

        with tempfile.TemporaryDirectory() as tmpdir:
            model_path = os.path.join(tmpdir, "model.pkl")
            joblib.dump(model, model_path)
            artifact = wandb.Artifact(
                name=f"{run_name.lower()}_estimator",
                type="model",
                metadata={"MSE": float(mse), "R2": float(r2), "run_name": run_name},
            )
            artifact.add_file(model_path, name="model.pkl")
            run.log_artifact(artifact)

MSE,▁
R2,▁
rmse,▁
MSE,1.74696
R2,0.97029
rmse,1.32173


MSE,▁
R2,▁
rmse,▁
MSE,1.75652
R2,0.97013
rmse,1.32534


MSE,▁
R2,▁
rmse,▁
MSE,1.71027
R2,0.97092
rmse,1.30777


MSE,▁
R2,▁
rmse,▁
MSE,3.67261
R2,0.93754
rmse,1.9164


MSE,▁
R2,▁
rmse,▁
MSE,0.90545
R2,0.9846
rmse,0.95155


MSE,▁
R2,▁
rmse,▁
MSE,0.97247
R2,0.98346
rmse,0.98614


MSE,▁
R2,▁
rmse,▁
MSE,1.77522
R2,0.96981
rmse,1.33237


MSE,▁
R2,▁
rmse,▁
MSE,1.76583
R2,0.96997
rmse,1.32885


MSE,▁
R2,▁
rmse,▁
MSE,2.20124
R2,0.96257
rmse,1.48366


MSE,▁
R2,▁
rmse,▁
MSE,1.31117
R2,0.9777
rmse,1.14506


# Gradient Boosting

In [6]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import numpy as np
import os
import tempfile
import wandb

# Prepare data
df_clean = df.dropna(subset=["apparent_temperature_max"])

# Parameter sweep
lgb_training_params = [
    # --- Baseline ---
    {"run_name": "lgb_baseline",         "params": {"n_estimators": 200, "learning_rate": 0.1,  "max_depth": -1,  "num_leaves": 31,  "min_child_samples": 20, "subsample": 1.0,  "colsample_bytree": 1.0,  "reg_alpha": 0.0, "reg_lambda": 0.0}},

    # --- Sweep learning rate ---
    {"run_name": "lgb_lr0.01",           "params": {"n_estimators": 200, "learning_rate": 0.01, "max_depth": -1,  "num_leaves": 31,  "min_child_samples": 20, "subsample": 1.0,  "colsample_bytree": 1.0,  "reg_alpha": 0.0, "reg_lambda": 0.0}},
    {"run_name": "lgb_lr0.05",           "params": {"n_estimators": 200, "learning_rate": 0.05, "max_depth": -1,  "num_leaves": 31,  "min_child_samples": 20, "subsample": 1.0,  "colsample_bytree": 1.0,  "reg_alpha": 0.0, "reg_lambda": 0.0}},
    {"run_name": "lgb_lr0.3",            "params": {"n_estimators": 200, "learning_rate": 0.3,  "max_depth": -1,  "num_leaves": 31,  "min_child_samples": 20, "subsample": 1.0,  "colsample_bytree": 1.0,  "reg_alpha": 0.0, "reg_lambda": 0.0}},

    # --- Sweep num_leaves (controls tree complexity) ---
    {"run_name": "lgb_leaves15",         "params": {"n_estimators": 200, "learning_rate": 0.1,  "max_depth": -1,  "num_leaves": 15,  "min_child_samples": 20, "subsample": 1.0,  "colsample_bytree": 1.0,  "reg_alpha": 0.0, "reg_lambda": 0.0}},
    {"run_name": "lgb_leaves63",         "params": {"n_estimators": 200, "learning_rate": 0.1,  "max_depth": -1,  "num_leaves": 63,  "min_child_samples": 20, "subsample": 1.0,  "colsample_bytree": 1.0,  "reg_alpha": 0.0, "reg_lambda": 0.0}},
    {"run_name": "lgb_leaves127",        "params": {"n_estimators": 200, "learning_rate": 0.1,  "max_depth": -1,  "num_leaves": 127, "min_child_samples": 20, "subsample": 1.0,  "colsample_bytree": 1.0,  "reg_alpha": 0.0, "reg_lambda": 0.0}},

    # --- Sweep n_estimators ---
    {"run_name": "lgb_n100",             "params": {"n_estimators": 100, "learning_rate": 0.1,  "max_depth": -1,  "num_leaves": 31,  "min_child_samples": 20, "subsample": 1.0,  "colsample_bytree": 1.0,  "reg_alpha": 0.0, "reg_lambda": 0.0}},
    {"run_name": "lgb_n500",             "params": {"n_estimators": 500, "learning_rate": 0.1,  "max_depth": -1,  "num_leaves": 31,  "min_child_samples": 20, "subsample": 1.0,  "colsample_bytree": 1.0,  "reg_alpha": 0.0, "reg_lambda": 0.0}},
    {"run_name": "lgb_n1000",            "params": {"n_estimators": 1000,"learning_rate": 0.05, "max_depth": -1,  "num_leaves": 31,  "min_child_samples": 20, "subsample": 1.0,  "colsample_bytree": 1.0,  "reg_alpha": 0.0, "reg_lambda": 0.0}},

    # --- Sweep subsampling (reduces overfitting) ---
    {"run_name": "lgb_subsample0.7",     "params": {"n_estimators": 200, "learning_rate": 0.1,  "max_depth": -1,  "num_leaves": 31,  "min_child_samples": 20, "subsample": 0.7,  "colsample_bytree": 1.0,  "reg_alpha": 0.0, "reg_lambda": 0.0}},
    {"run_name": "lgb_colsample0.7",     "params": {"n_estimators": 200, "learning_rate": 0.1,  "max_depth": -1,  "num_leaves": 31,  "min_child_samples": 20, "subsample": 1.0,  "colsample_bytree": 0.7,  "reg_alpha": 0.0, "reg_lambda": 0.0}},
    {"run_name": "lgb_sub0.7_col0.7",   "params": {"n_estimators": 200, "learning_rate": 0.1,  "max_depth": -1,  "num_leaves": 31,  "min_child_samples": 20, "subsample": 0.7,  "colsample_bytree": 0.7,  "reg_alpha": 0.0, "reg_lambda": 0.0}},

    # --- Sweep regularization ---
    {"run_name": "lgb_l1_0.1",          "params": {"n_estimators": 200, "learning_rate": 0.1,  "max_depth": -1,  "num_leaves": 31,  "min_child_samples": 20, "subsample": 1.0,  "colsample_bytree": 1.0,  "reg_alpha": 0.1, "reg_lambda": 0.0}},
    {"run_name": "lgb_l2_0.1",          "params": {"n_estimators": 200, "learning_rate": 0.1,  "max_depth": -1,  "num_leaves": 31,  "min_child_samples": 20, "subsample": 1.0,  "colsample_bytree": 1.0,  "reg_alpha": 0.0, "reg_lambda": 0.1}},
    {"run_name": "lgb_l1_l2",           "params": {"n_estimators": 200, "learning_rate": 0.1,  "max_depth": -1,  "num_leaves": 31,  "min_child_samples": 20, "subsample": 1.0,  "colsample_bytree": 1.0,  "reg_alpha": 0.1, "reg_lambda": 0.1}},
]

for config in lgb_training_params:
    run_name = config["run_name"]
    params   = config["params"]

    with wandb.init(project="weather-prediction", name=run_name, config=params) as run:
        model = lgb.LGBMRegressor(
            **params,
            random_state=42,
            n_jobs=-1,
            verbose=-1       # suppress LightGBM's per-tree console output
        )

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        mse  = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2   = r2_score(y_test, y_pred)

        importances = dict(zip(X.columns, model.feature_importances_.round(4)))

        run.log({
            "MSE":                 mse,
            "R2":                  r2,
            "rmse":                rmse,
            "feature_importances": importances,
        })

        with tempfile.TemporaryDirectory() as tmpdir:
            model_path = os.path.join(tmpdir, "model.pkl")
            joblib.dump(model, model_path)
            artifact = wandb.Artifact(
                name=f"{run_name.lower()}_estimator",
                type="model",
                metadata={"MSE": float(mse), "R2": float(r2), "run_name": run_name},
            )
            artifact.add_file(model_path, name="model.pkl")
            run.log_artifact(artifact)

MSE,▁
R2,▁
rmse,▁
MSE,0.56577
R2,0.99038
rmse,0.75218


MSE,▁
R2,▁
rmse,▁
MSE,2.04174
R2,0.96528
rmse,1.42889


MSE,▁
R2,▁
rmse,▁
MSE,0.56978
R2,0.99031
rmse,0.75484


MSE,▁
R2,▁
rmse,▁
MSE,0.60857
R2,0.98965
rmse,0.78011


MSE,▁
R2,▁
rmse,▁
MSE,0.61934
R2,0.98947
rmse,0.78698


...

MSE,▁
R2,▁
rmse,▁
MSE,0.61035
R2,0.98962
rmse,0.78125


MSE,▁
R2,▁
rmse,▁
MSE,0.61035
R2,0.98962
rmse,0.78125


MSE,▁
R2,▁
rmse,▁
MSE,0.58054
R2,0.99013
rmse,0.76193


MSE,▁
R2,▁
rmse,▁
MSE,0.57062
R2,0.9903
rmse,0.7554


MSE,▁
R2,▁
rmse,▁
MSE,0.57653
R2,0.9902
rmse,0.75929


## Fake validation data (Faker)

Rows are generated with [Faker](https://faker.readthedocs.io/) dates and uniform random values **within each column’s min/max** on the real preprocessed data, so scales look similar. **Targets are independent of features**, so metrics here are mainly a sanity check (pipeline, shapes, `predict`) rather than generalization quality. For that, use the real holdout `X_test, y_test`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

from scripts.fake_weather_data import (
    align_feature_columns,
    generate_fake_weather,
    preprocess_like_training_notebook,
)

df_template = pd.read_pickle("../data/processed/preprocessed_weather.pkl")
fake_raw = generate_fake_weather(df_template, n_rows=200, seed=123)
df_fake = preprocess_like_training_notebook(fake_raw)

X_ref = df.drop(columns=["apparent_temperature_max"]).columns
X_fake = df_fake.drop(columns=["apparent_temperature_max"])
X_fake = align_feature_columns(X_fake, X_ref)
y_fake = df_fake["apparent_temperature_max"]

# Same split as earlier cells (requires the first preprocessing cell to have run on ``df``).
X_all = df.drop(columns=["apparent_temperature_max"])
y_all = df["apparent_temperature_max"]
X_train_local, _, y_train_local, _ = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42
)

lin = LinearRegression()
lin.fit(X_train_local, y_train_local)
y_pred_fake = lin.predict(X_fake)

print(f"Fake validation MSE: {mean_squared_error(y_fake, y_pred_fake):.4f}")
print(f"Fake validation R²:  {r2_score(y_fake, y_pred_fake):.4f}")
pd.DataFrame({"y_true": y_fake.values, "y_pred": y_pred_fake}).head(10)

In [8]:
# Load the best model from W&B
import wandb
import joblib

# Initialize W&B
run = wandb.init(entity="weatherml", project="weather-prediction")
artifact = run.use_artifact("weatherml/weather-prediction/lgb_lr0.01_estimator:v0", type="model")
artifact_dir = artifact.download()

print(artifact)

wandb:   1 of 1 files downloaded.  


<Artifact QXJ0aWZhY3Q6MjcwNjQ0NTE4OQ==>


In [9]:
# Load the model from the artifact
model = joblib.load(os.path.join(artifact_dir, "model.pkl"))

print(model)

LGBMRegressor(learning_rate=0.01, n_estimators=200, n_jobs=-1, random_state=42,
              verbose=-1)


In [10]:
# Use the model to make predictions (AKA "inference")
y_pred = model.predict(X_test)

print(y_pred)


[39.48657187 37.03492554 31.48796835 26.12586315 42.11714264 39.61805117
 30.76999315 37.05191761 29.26897874 37.12321888 30.93298653 38.85190753
 39.47375485 40.33525051 39.02646471 37.79270613 21.71141347 36.67740181
 36.63498222 38.10168392 36.79184191 39.28719325 36.86049552 27.92301612
 32.17315376 39.55003273 24.6778988  27.94235838 27.78088497 33.27772173
 35.94109074 31.4320066  20.7718399  38.79724102 36.68578323 27.65402906
 32.78330325 26.98690537 28.70896697 29.36434974 25.63393189 38.85865553
 37.19704582 23.34702017 26.83676508 35.80018494 37.58906047 40.59681587
 19.63752967 20.54154847 29.38808955 40.41646575 37.20327833 38.73025872
 25.50650591 37.54684633 41.0902287  42.5992182  27.99470779 27.01935232
 18.5007637  31.09716067 37.70044525 22.28091687 38.12491293 42.59955403
 41.46213072 24.6598752  32.94543411 40.15645329 32.47055387 38.34316385
 37.58371768 20.63406015 35.90955241 23.73026237 26.82029763 35.07233525
 36.62563903 27.01443769 18.27143088 31.82795479 36

Index(['temperature_2m_max', 'temperature_2m_min', 'temperature_2m_mean',
       'apparent_temperature_min', 'relative_humidity_2m_max',
       'relative_humidity_2m_min', 'relative_humidity_2m_mean',
       'wind_speed_10m_max', 'wind_gusts_10m_max',
       'wind_direction_10m_dominant', 'precipitation_sum', 'rain_sum',
       'snowfall_sum', 'precipitation_hours', 'shortwave_radiation_sum',
       'sunshine_duration', 'target', 'year', 'month', 'day', 'weather_code_1',
       'weather_code_2', 'weather_code_3', 'weather_code_51',
       'weather_code_53', 'weather_code_55', 'weather_code_61',
       'weather_code_63', 'weather_code_65'],
      dtype='object')
  temperature_2m_max temperature_2m_min temperature_2m_mean  \
0          31.144384          20.408493           25.390685   

  apparent_temperature_min relative_humidity_2m_max relative_humidity_2m_min  \
0                22.109041                84.435616                44.120548   

  relative_humidity_2m_mean wind_speed_10m

ValueError: pandas dtypes must be int, float or bool.
Fields with bad pandas dtypes: temperature_2m_max: object, temperature_2m_min: object, temperature_2m_mean: object, apparent_temperature_min: object, relative_humidity_2m_max: object, relative_humidity_2m_min: object, relative_humidity_2m_mean: object, wind_speed_10m_max: object, wind_gusts_10m_max: object, wind_direction_10m_dominant: object, precipitation_sum: object, rain_sum: object, snowfall_sum: object, precipitation_hours: object, shortwave_radiation_sum: object, sunshine_duration: object, target: object, year: object, month: object, day: object, weather_code_1: object, weather_code_2: object, weather_code_3: object, weather_code_51: object, weather_code_53: object, weather_code_55: object, weather_code_61: object, weather_code_63: object, weather_code_65: object